
<div style="background: linear-gradient(135deg,#0B3D91,#0F766E); padding:28px; border-radius:16px; box-shadow:0px 10px 26px rgba(0,0,0,0.18);">
  <h1 style="color:white; text-align:center; font-size:36px; margin:0;">
    Deep and Generative AI Analytics and Open-Source Metrics Design
  </h1>
  <p style="color:white; text-align:center; font-size:16px; margin:10px 0 0 0;">
    A comprehensively detailed, operable, functionally resourceful notebook package
  </p>
  <p style="color:white; text-align:center; font-size:14px; margin:8px 0 0 0; opacity:0.92;">
    By DaScient
  </p>
</div>



## Scope and intent

This notebook is an operational blueprint for building a metrics and analytics system for deep learning and generative AI.

It is designed to be:

- **Operable**: executable code paths that run end to end using synthetic data by default  
- **Extensible**: pluggable metrics, datasets, and model adapters  
- **Auditable**: deterministic seeds, dataset hashing, versioned metric definitions, structured logs  
- **Educational**: full prose explanations for each design decision

Where the notebook references optional heavyweight tooling (Transformers, diffusion, FID, GPU profilers), it provides guarded cells that only run if the dependencies are available.

No part of the system relies on hidden state. Everything you need is defined in cells, versionable, and exportable.



## Table of contents

1. System principles and evaluation ontology  
2. Environment setup and reproducibility controls  
3. Configuration schema (single source of truth)  
4. Dataset integrity and observability (profiles, drift, shift)  
5. Metric engine design (registry, versioning, tests)  
6. Training analytics hooks (gradients, activations, stability)  
7. Generative evaluation: text and image (operable baselines, optional advanced)  
8. Composite scoring (transparent weighting)  
9. Reporting: JSONL logs and a minimal dashboard  
10. Governance: model cards, metric audits, release checklist  
11. Packaging: exporting the core as a small local module



# 1. System principles and evaluation ontology

A metrics system is not a scoreboard. It is a set of instruments.

A well designed instrument:

- measures the intended quantity, not a proxy that happens to correlate in one dataset  
- exposes its assumptions  
- survives replication  
- can be falsified by controlled failure cases  

For deep and generative systems, analytics must be multi axis:

- **Performance**: task success under nominal conditions  
- **Robustness**: stability under shift, perturbation, and noise  
- **Calibration**: whether confidence aligns with truth  
- **Efficiency**: cost, latency, memory, energy per unit output quality  
- **Safety and alignment**: refusal quality, toxic content rates, policy compliance signals  
- **Reproducibility**: deterministic controls, data hashes, metric versioning

Open-source metrics design adds two more constraints:

- **Portability**: minimal assumptions about a specific lab environment  
- **Transparency**: metric definitions must be explicit, versioned, and testable



# 2. Environment setup and reproducibility controls

This section provides:

- deterministic seeds  
- strict logging conventions  
- platform fingerprints (Python, package versions)  
- a standard project directory layout

If you are running this notebook in a fresh environment, run the optional install cell first.


In [ ]:

# Optional installs. Uncomment if you need them.
# Notes:
# - Keep dependencies minimal by default.
# - Heavyweight dependencies are optional and guarded later.

# %pip install -q numpy pandas scikit-learn matplotlib pyyaml
# %pip install -q torch --index-url https://download.pytorch.org/whl/cpu
# %pip install -q transformers datasets evaluate accelerate


In [ ]:

import os, sys, json, time, math, hashlib, platform, random
from dataclasses import dataclass, asdict
from typing import Dict, Any, List, Optional, Callable, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

def set_global_seed(seed: int) -> None:
    """Set seeds across common RNG sources for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

    # Torch is optional
    try:
        import torch
        torch.manual_seed(seed)
        torch.use_deterministic_algorithms(True)
    except Exception:
        pass

def fingerprint_environment() -> Dict[str, Any]:
    """Return a minimal environment fingerprint useful for audit logs."""
    fp = {
        "python": sys.version,
        "platform": platform.platform(),
        "executable": sys.executable,
        "timestamp_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    # Package versions (best effort)
    try:
        import sklearn
        fp["sklearn"] = sklearn.__version__
    except Exception:
        fp["sklearn"] = None
    fp["numpy"] = np.__version__
    fp["pandas"] = pd.__version__
    fp["matplotlib"] = plt.matplotlib.__version__
    return fp

set_global_seed(42)
env_fp = fingerprint_environment()
env_fp



# 3. Configuration schema (single source of truth)

Operational analytics fails when configuration is scattered across ad hoc cells.

We define a single configuration object that drives:

- dataset generation and splits  
- metric selection and parameters  
- run identifiers and logging paths  
- composite score weights  
- drift thresholds and alerting rules

This configuration is serializable and hashable. It becomes the root of reproducibility.


In [ ]:

@dataclass(frozen=True)
class RunConfig:
    project_name: str = "dascient_deepgenai_metrics"
    run_name: str = "baseline"
    seed: int = 42

    # Data
    n_samples: int = 6000
    n_features: int = 20
    test_size: float = 0.25
    drift_test_alpha: float = 0.01  # KS test significance for drift alerts

    # Metrics (names map to registry keys)
    classification_metrics: Tuple[str, ...] = ("accuracy", "f1_weighted", "auc")
    calibration_metrics: Tuple[str, ...] = ("log_loss",)
    robustness_metrics: Tuple[str, ...] = ("shift_sensitivity",)

    # Composite score weights
    composite_weights: Dict[str, float] = None

    # Logging
    output_dir: str = "./runs"
    jsonl_log_name: str = "metrics.jsonl"

    def __post_init__(self):
        if self.composite_weights is None:
            object.__setattr__(self, "composite_weights", {
                "accuracy": 0.35,
                "f1_weighted": 0.25,
                "auc": 0.15,
                "log_loss": -0.15,  # negative weight because lower is better
                "shift_sensitivity": -0.10
            })

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def stable_hash(self) -> str:
        blob = json.dumps(self.to_dict(), sort_keys=True).encode("utf-8")
        return hashlib.sha256(blob).hexdigest()[:16]

cfg = RunConfig()
cfg.to_dict(), cfg.stable_hash()



## Standard run directory layout

Each run writes to a folder named by:

- project name  
- run name  
- config hash

This keeps experiments immutable and easy to diff.


In [ ]:

from pathlib import Path

def make_run_dir(cfg: RunConfig) -> Path:
    run_dir = Path(cfg.output_dir) / cfg.project_name / f"{cfg.run_name}_{cfg.stable_hash()}"
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir

run_dir = make_run_dir(cfg)
run_dir



# 4. Dataset integrity and observability (profiles, drift, shift)

We do three things:

1. **Integrity**: compute a dataset hash so we can detect accidental changes  
2. **Profiling**: produce feature level summaries and missingness  
3. **Shift**: simulate distribution shift and quantify sensitivity

By default, this notebook generates synthetic tabular data. Replace the generator with your own dataset loader later.


In [ ]:

def make_synthetic_classification(n_samples: int, n_features: int, seed: int):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(n_samples, n_features))

    # Nonlinear signal in first few features
    logits = 1.2 * X[:, 0] - 0.8 * X[:, 1] + 0.5 * (X[:, 2] ** 2) - 0.3 * X[:, 3]
    probs = 1 / (1 + np.exp(-logits))
    y = rng.binomial(1, probs)

    df = pd.DataFrame(X, columns=[f"f{i:02d}" for i in range(n_features)])
    return df, pd.Series(y, name="y")

def dataframe_hash(df: pd.DataFrame, y: pd.Series = None) -> str:
    """Hash data content deterministically. Suitable for audit logs."""
    hasher = hashlib.sha256()
    hasher.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
    if y is not None:
        hasher.update(pd.util.hash_pandas_object(y, index=True).values.tobytes())
    return hasher.hexdigest()[:16]

X_df, y = make_synthetic_classification(cfg.n_samples, cfg.n_features, cfg.seed)
data_hash = dataframe_hash(X_df, y)
data_hash, X_df.shape, y.value_counts(normalize=True).to_dict()


In [ ]:

def profile_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        s = df[col]
        rows.append({
            "feature": col,
            "dtype": str(s.dtype),
            "mean": float(s.mean()),
            "std": float(s.std(ddof=1)),
            "min": float(s.min()),
            "max": float(s.max()),
            "missing_pct": float(s.isna().mean()),
            "unique": int(s.nunique()),
            "p01": float(np.quantile(s, 0.01)),
            "p50": float(np.quantile(s, 0.50)),
            "p99": float(np.quantile(s, 0.99)),
        })
    return pd.DataFrame(rows)

profile_df = profile_dataframe(X_df)
profile_df.head(8)



## Drift testing

We implement two practical drift detectors for numerical features:

- **KS test**: detects distribution differences in a nonparametric way  
- **PSI (Population Stability Index)**: common in risk modeling, interpretable thresholds

These are not perfect. They are operational tools, best used with dashboards and human review.


In [ ]:

from scipy.stats import ks_2samp

def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10, eps: float = 1e-8) -> float:
    """Population Stability Index for one feature."""
    q = np.linspace(0, 1, bins + 1)
    breaks = np.quantile(expected, q)
    breaks[0] = -np.inf
    breaks[-1] = np.inf

    e_counts, _ = np.histogram(expected, bins=breaks)
    a_counts, _ = np.histogram(actual, bins=breaks)

    e_perc = (e_counts / max(e_counts.sum(), 1)).astype(float) + eps
    a_perc = (a_counts / max(a_counts.sum(), 1)).astype(float) + eps

    return float(np.sum((a_perc - e_perc) * np.log(a_perc / e_perc)))

def drift_report(df_ref: pd.DataFrame, df_cur: pd.DataFrame, alpha: float = 0.01) -> pd.DataFrame:
    rows = []
    for col in df_ref.columns:
        ref = df_ref[col].to_numpy()
        cur = df_cur[col].to_numpy()
        ks = ks_2samp(ref, cur, alternative="two-sided", mode="auto")
        rows.append({
            "feature": col,
            "ks_stat": float(ks.statistic),
            "ks_pvalue": float(ks.pvalue),
            "ks_alert": bool(ks.pvalue < alpha),
            "psi": psi(ref, cur, bins=10),
        })
    out = pd.DataFrame(rows).sort_values(["ks_alert", "psi"], ascending=[False, False]).reset_index(drop=True)
    return out

# Create a shifted copy (simulate real world drift)
X_shift = X_df.copy()
X_shift["f00"] = X_shift["f00"] + 0.65   # mean shift
X_shift["f01"] = X_shift["f01"] * 1.35   # variance shift

drift_df = drift_report(X_df, X_shift, alpha=cfg.drift_test_alpha)
drift_df.head(10)



# 5. Metric engine design (registry, versioning, tests)

Open-source metrics design requires:

- a registry keyed by stable names  
- explicit version identifiers for each metric definition  
- minimal unit tests for sanity and edge cases  
- consistent directionality (higher is better or lower is better)

We implement a small metric engine that:

- computes metrics from a shared payload  
- writes JSONL records for each evaluation stage  
- supports per metric metadata: version, direction, and docs


In [ ]:

@dataclass(frozen=True)
class MetricSpec:
    key: str
    version: str
    higher_is_better: bool
    doc: str
    fn: Callable[[Dict[str, Any]], float]

class MetricRegistry:
    def __init__(self):
        self._specs: Dict[str, MetricSpec] = {}

    def register(self, spec: MetricSpec) -> None:
        if spec.key in self._specs:
            raise ValueError(f"Metric already registered: {spec.key}")
        self._specs[spec.key] = spec

    def get(self, key: str) -> MetricSpec:
        if key not in self._specs:
            raise KeyError(f"Unknown metric: {key}")
        return self._specs[key]

    def keys(self) -> List[str]:
        return sorted(self._specs.keys())

registry = MetricRegistry()


In [ ]:

def require(payload: Dict[str, Any], *names: str):
    for n in names:
        if n not in payload:
            raise KeyError(f"Missing '{n}' in payload")

def m_accuracy(payload: Dict[str, Any]) -> float:
    require(payload, "y_true", "y_pred")
    return float(accuracy_score(payload["y_true"], payload["y_pred"]))

def m_f1_weighted(payload: Dict[str, Any]) -> float:
    require(payload, "y_true", "y_pred")
    return float(f1_score(payload["y_true"], payload["y_pred"], average="weighted"))

def m_auc(payload: Dict[str, Any]) -> float:
    require(payload, "y_true", "y_prob")
    y_prob = np.asarray(payload["y_prob"])
    if np.allclose(y_prob, y_prob[0]):
        return float("nan")
    return float(roc_auc_score(payload["y_true"], y_prob))

def m_log_loss(payload: Dict[str, Any]) -> float:
    require(payload, "y_true", "y_prob")
    y_prob = np.clip(np.asarray(payload["y_prob"]), 1e-8, 1 - 1e-8)
    return float(log_loss(payload["y_true"], y_prob))

def m_shift_sensitivity(payload: Dict[str, Any]) -> float:
    """A simple robustness proxy: delta in accuracy under a defined shift set."""
    require(payload, "acc_ref", "acc_shift")
    return float(payload["acc_ref"] - payload["acc_shift"])

registry.register(MetricSpec(
    key="accuracy",
    version="1.0.0",
    higher_is_better=True,
    doc="Accuracy on the evaluation set.",
    fn=m_accuracy
))
registry.register(MetricSpec(
    key="f1_weighted",
    version="1.0.0",
    higher_is_better=True,
    doc="Weighted F1 score, robust to class imbalance.",
    fn=m_f1_weighted
))
registry.register(MetricSpec(
    key="auc",
    version="1.0.0",
    higher_is_better=True,
    doc="ROC AUC computed from predicted probabilities. Returns NaN for degenerate probabilities.",
    fn=m_auc
))
registry.register(MetricSpec(
    key="log_loss",
    version="1.0.0",
    higher_is_better=False,
    doc="Log loss (cross entropy) from predicted probabilities. Lower is better.",
    fn=m_log_loss
))
registry.register(MetricSpec(
    key="shift_sensitivity",
    version="1.0.0",
    higher_is_better=False,
    doc="Robustness proxy: accuracy drop between reference and shifted data.",
    fn=m_shift_sensitivity
))

registry.keys()



## Sanity tests

Metrics should fail loudly when required fields are absent, and handle obvious edge cases.

These tests are small, but they prevent quiet corruption.


In [ ]:

def run_metric_sanity_tests():
    payload = {
        "y_true": np.array([0, 1, 1, 0]),
        "y_pred": np.array([0, 1, 0, 0]),
        "y_prob": np.array([0.2, 0.9, 0.4, 0.3]),
        "acc_ref": 0.90,
        "acc_shift": 0.84,
    }
    out = {}
    for k in registry.keys():
        spec = registry.get(k)
        out[k] = spec.fn(payload)

    assert 0 <= out["accuracy"] <= 1
    assert 0 <= out["f1_weighted"] <= 1
    assert 0 <= out["auc"] <= 1
    assert out["log_loss"] >= 0
    assert out["shift_sensitivity"] >= 0

    payload2 = payload.copy()
    payload2["y_prob"] = np.array([0.5, 0.5, 0.5, 0.5])
    assert math.isnan(registry.get("auc").fn(payload2))

    return out

run_metric_sanity_tests()



# 6. Training analytics hooks (gradients, activations, stability)

A metrics layer measures outputs. A training analytics layer measures the learning process.

For deep systems, the most operational hooks are:

- gradient norms (detect vanishing and exploding updates)  
- weight update magnitudes  
- activation sparsity or saturation proxies  
- learning rate schedules and optimizer state summaries  

In this notebook we implement a lightweight, framework agnostic interface plus an optional PyTorch adapter.


In [ ]:

def basic_stability_diagnostics(loss_history):
    loss_arr = np.asarray(loss_history, dtype=float)
    if len(loss_arr) < 3:
        return {"n": len(loss_arr)}
    deltas = np.diff(loss_arr)
    return {
        "n": int(len(loss_arr)),
        "loss_min": float(loss_arr.min()),
        "loss_max": float(loss_arr.max()),
        "loss_last": float(loss_arr[-1]),
        "delta_mean": float(deltas.mean()),
        "delta_std": float(deltas.std(ddof=1)),
        "non_monotonic_steps": int(np.sum(deltas > 0)),
    }

loss_hist = list(np.exp(-np.linspace(0, 4, 40)) + 0.03*np.random.randn(40))
basic_stability_diagnostics(loss_hist)



## Optional: PyTorch gradient hook

This cell defines a helper that collects parameter gradient norms during a backward pass.

It is safe to run even if torch is not installed. If torch is absent, it will print a message and do nothing.


In [ ]:

def torch_gradient_norms(model):
    try:
        import torch
    except Exception:
        print("Torch not available. Skipping gradient hook.")
        return None

    rows = []
    for name, p in model.named_parameters():
        if p.grad is None:
            continue
        rows.append({"param": name, "grad_norm": float(p.grad.detach().norm().cpu().item())})
    return pd.DataFrame(rows).sort_values("grad_norm", ascending=False).reset_index(drop=True)

try:
    import torch
    import torch.nn as nn

    set_global_seed(cfg.seed)
    m = nn.Sequential(nn.Linear(cfg.n_features, 16), nn.ReLU(), nn.Linear(16, 2))
    x = torch.randn(64, cfg.n_features)
    yb = torch.randint(0, 2, (64,))
    logits = m(x)
    loss = nn.CrossEntropyLoss()(logits, yb)
    loss.backward()
    torch_gradient_norms(m).head(10)
except Exception as e:
    print("Torch demo skipped:", str(e)[:180])



# 7. Operable evaluation baseline (tabular classifier)

We build a simple baseline model to demonstrate:

- consistent train/test splitting  
- prediction payload creation  
- metric computation via registry  
- structured logging to JSONL  
- robustness by evaluation on shifted data  

Replace the baseline with your deep model later. The infrastructure remains the same.


In [ ]:

set_global_seed(cfg.seed)

X_train, X_test, y_train, y_test = train_test_split(
    X_df, y, test_size=cfg.test_size, random_state=cfg.seed, stratify=y
)

clf = LogisticRegression(max_iter=400)
clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

payload = {
    "y_true": y_test.to_numpy(),
    "y_pred": y_pred,
    "y_prob": y_prob
}

ref_metrics = {}
for k in cfg.classification_metrics + cfg.calibration_metrics:
    spec = registry.get(k)
    ref_metrics[k] = spec.fn(payload)

ref_metrics


In [ ]:

X_test_shift = X_test.copy()
X_test_shift["f00"] = X_test_shift["f00"] + 0.65
X_test_shift["f01"] = X_test_shift["f01"] * 1.35

y_prob_shift = clf.predict_proba(X_test_shift)[:, 1]
y_pred_shift = (y_prob_shift >= 0.5).astype(int)

payload_shift = {
    "y_true": y_test.to_numpy(),
    "y_pred": y_pred_shift,
    "y_prob": y_prob_shift
}

acc_ref = registry.get("accuracy").fn(payload)
acc_shift = registry.get("accuracy").fn(payload_shift)

rob_payload = {"acc_ref": acc_ref, "acc_shift": acc_shift}
rob_metrics = {"shift_sensitivity": registry.get("shift_sensitivity").fn(rob_payload)}

acc_ref, acc_shift, rob_metrics



## Structured logging (JSONL)

Every evaluation produces a record with:

- run config hash  
- dataset hash  
- environment fingerprint  
- metric key and version  
- metric value  
- stage name (train, test, shift, safety, etc.)

This makes downstream dashboards trivial.


In [ ]:

def jsonl_append(path: str, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, sort_keys=True) + "\n")

def log_metrics(stage: str, metrics, cfg: RunConfig, run_dir) -> str:
    out_path = str(Path(run_dir) / cfg.jsonl_log_name)
    base = {
        "project": cfg.project_name,
        "run_name": cfg.run_name,
        "config_hash": cfg.stable_hash(),
        "dataset_hash": data_hash,
        "stage": stage,
        "env": env_fp,
        "ts_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    for k, v in metrics.items():
        spec = registry.get(k) if k in registry.keys() else None
        rec = dict(base)
        rec.update({
            "metric": k,
            "metric_version": spec.version if spec else None,
            "higher_is_better": spec.higher_is_better if spec else None,
            "value": None if (v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v)))) else float(v),
        })
        jsonl_append(out_path, rec)
    return out_path

log_path = log_metrics("test_ref", ref_metrics, cfg, run_dir)
log_path = log_metrics("test_shift", {"accuracy": acc_shift, **rob_metrics}, cfg, run_dir)

log_path



# 8. Reporting: minimal dashboard

We render:

- a drift report table  
- a metric table for reference vs shift  
- a simple composite score (transparent weights)

This is not a full product dashboard. It is the minimal operable seed.


In [ ]:

def composite_score(metrics: Dict[str, float], weights: Dict[str, float]) -> float:
    total = 0.0
    for k, w in weights.items():
        if k not in metrics:
            continue
        v = metrics[k]
        if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
            continue
        total += w * float(v)
    return float(total)

metric_table = pd.DataFrame([
    {"metric": "accuracy", "ref": acc_ref, "shift": acc_shift},
    {"metric": "f1_weighted", "ref": ref_metrics.get("f1_weighted"), "shift": registry.get("f1_weighted").fn(payload_shift)},
    {"metric": "auc", "ref": ref_metrics.get("auc"), "shift": registry.get("auc").fn(payload_shift)},
    {"metric": "log_loss", "ref": ref_metrics.get("log_loss"), "shift": registry.get("log_loss").fn(payload_shift)},
    {"metric": "shift_sensitivity", "ref": rob_metrics["shift_sensitivity"], "shift": rob_metrics["shift_sensitivity"]},
])

ref_all = {
    "accuracy": acc_ref,
    "f1_weighted": ref_metrics.get("f1_weighted"),
    "auc": ref_metrics.get("auc"),
    "log_loss": ref_metrics.get("log_loss"),
    "shift_sensitivity": rob_metrics["shift_sensitivity"],
}
ref_cgis = composite_score(ref_all, cfg.composite_weights)

metric_table, ref_cgis


In [ ]:

top = drift_df.head(12).copy()

plt.figure(figsize=(10, 4))
plt.bar(top["feature"], top["psi"])
plt.xticks(rotation=45, ha="right")
plt.title("Top feature drift by PSI (higher indicates more shift)")
plt.ylabel("PSI")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
plt.bar(metric_table["metric"], metric_table["ref"])
plt.xticks(rotation=45, ha="right")
plt.title("Reference metrics (test set)")
plt.ylabel("value")
plt.tight_layout()
plt.show()



# 9. Generative evaluation: operable baselines and optional advanced adapters

Generative systems require evaluation across multiple axes.

This notebook provides two tracks:

1. **Operable baselines** that run now:
   - lexical diversity metrics
   - repetition metrics
   - length normalized entropy proxies
   - a simple safety placeholder interface (user supplied classifier)

2. **Optional advanced adapters**:
   - Transformers perplexity evaluation
   - embedding similarity (BERTScore style)
   - image metrics stubs (FID, CLIPScore) guarded by availability

The key point is not a single metric. It is a reproducible harness where metrics can be swapped without rewriting the pipeline.


In [ ]:

import re
from collections import Counter

def normalize_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def token_stats(texts):
    toks = []
    for t in texts:
        toks.extend(normalize_text(t).split(" "))

    toks = [x for x in toks if x]
    if not toks:
        return {"n_tokens": 0, "vocab": 0, "type_token_ratio": float("nan"), "rep_2gram": float("nan")}

    vocab = len(set(toks))
    ttr = vocab / len(toks)

    bigrams = list(zip(toks[:-1], toks[1:])) if len(toks) >= 2 else []
    if not bigrams:
        rep = 0.0
    else:
        c = Counter(bigrams)
        rep = 1 - (len(c) / len(bigrams))

    return {
        "n_tokens": float(len(toks)),
        "vocab": float(vocab),
        "type_token_ratio": float(ttr),
        "rep_2gram": float(rep),
    }

generations = [
    "The system measures performance, robustness, and calibration.",
    "A good metric reveals intent and survives replication.",
    "Analytics should clarify, not decorate."
]
token_stats(generations)



## Optional: Perplexity with Transformers (guarded)

If you have `transformers` and a local model available, you can compute perplexity on a text set.

This cell is guarded and will not fail the notebook if dependencies are absent.


In [ ]:

def transformers_perplexity(texts, model_name: str = "gpt2", max_texts: int = 32):
    try:
        import torch
        from transformers import AutoTokenizer, AutoModelForCausalLM
    except Exception:
        print("transformers or torch not available. Skipping perplexity.")
        return None

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.eval()

    losses = []
    with torch.no_grad():
        for t in texts[:max_texts]:
            enc = tokenizer(t, return_tensors="pt")
            out = model(**enc, labels=enc["input_ids"])
            losses.append(float(out.loss.item()))
    if not losses:
        return None
    return float(math.exp(float(np.mean(losses))))

# Example usage (may download weights if not cached)
# ppl = transformers_perplexity(generations, model_name="gpt2")
# ppl



# 10. Open-source metrics design checklist

For a public metrics repository, enforce these operational rules:

- Metric definitions must be in code, not only in papers  
- Every metric has:
  - a stable key  
  - a semantic version  
  - a docstring that states assumptions and failure modes  
  - unit tests  
- Evaluation is configuration driven and logged to JSONL  
- Datasets are hashed, with provenance documented  
- Composite scores never hide components  
- Releases include:
  - model card style summary  
  - drift and robustness results  
  - efficiency measurements (latency, memory, cost)  

Below we generate a minimal model card and a run report in markdown and save them to the run directory.


In [ ]:

from pathlib import Path

def write_text(path: str, text: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

model_card = f"""
# Model Card (Template)

## Model name
{cfg.project_name}:{cfg.run_name}

## Intended use
Analytics and metrics demonstration baseline. Replace the model adapter to evaluate your system.

## Training data
Synthetic generator in notebook. Dataset hash: {data_hash}

## Evaluation summary
Reference accuracy: {acc_ref:.4f}
Shifted accuracy: {acc_shift:.4f}
Shift sensitivity (accuracy drop): {rob_metrics['shift_sensitivity']:.4f}
Composite score (weighted): {ref_cgis:.4f}

## Limitations
- Synthetic data is not a real world distribution.
- Robustness proxy is a simple shift simulation.
- Generative evaluation section contains operable baselines and optional advanced adapters.

## Reproducibility
Config hash: {cfg.stable_hash()}
Environment fingerprint: {json.dumps(env_fp, indent=2)}
"""

report_md_text = f"""
# Run Report

- Project: {cfg.project_name}
- Run name: {cfg.run_name}
- Config hash: {cfg.stable_hash()}
- Dataset hash: {data_hash}
- Run directory: {str(run_dir)}

## Metrics (reference)
{metric_table.to_markdown(index=False)}
"""

model_card_path = str(Path(run_dir) / "MODEL_CARD.md")
report_path = str(Path(run_dir) / "RUN_REPORT.md")
write_text(model_card_path, model_card.strip() + "\n")
write_text(report_path, report_md_text.strip() + "\n")

model_card_path, report_path



# 11. Packaging: export core as a small local module

Notebooks are good for exploration. Operational systems prefer importable modules.

This section exports a minimal `metrics_core.py` into the run directory containing:

- configuration dataclass  
- drift utilities  
- JSONL logger stub  

You can then import it elsewhere or convert it into a proper package.


In [ ]:

metrics_core_py = r"""
# metrics_core.py
# By DaScient
# Minimal export from the notebook for reuse.

import os, sys, json, time, math, hashlib, random
from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

def dataframe_hash(df: pd.DataFrame, y: Optional[pd.Series] = None) -> str:
    hasher = hashlib.sha256()
    hasher.update(pd.util.hash_pandas_object(df, index=True).values.tobytes())
    if y is not None:
        hasher.update(pd.util.hash_pandas_object(y, index=True).values.tobytes())
    return hasher.hexdigest()[:16]

def psi(expected: np.ndarray, actual: np.ndarray, bins: int = 10, eps: float = 1e-8) -> float:
    q = np.linspace(0, 1, bins + 1)
    breaks = np.quantile(expected, q)
    breaks[0] = -np.inf
    breaks[-1] = np.inf
    e_counts, _ = np.histogram(expected, bins=breaks)
    a_counts, _ = np.histogram(actual, bins=breaks)
    e_perc = (e_counts / max(e_counts.sum(), 1)).astype(float) + eps
    a_perc = (a_counts / max(a_counts.sum(), 1)).astype(float) + eps
    return float(np.sum((a_perc - e_perc) * np.log(a_perc / e_perc)))

def drift_report(df_ref: pd.DataFrame, df_cur: pd.DataFrame, alpha: float = 0.01) -> pd.DataFrame:
    rows = []
    for col in df_ref.columns:
        ref = df_ref[col].to_numpy()
        cur = df_cur[col].to_numpy()
        ks = ks_2samp(ref, cur, alternative="two-sided", mode="auto")
        rows.append({
            "feature": col,
            "ks_stat": float(ks.statistic),
            "ks_pvalue": float(ks.pvalue),
            "ks_alert": bool(ks.pvalue < alpha),
            "psi": psi(ref, cur, bins=10),
        })
    out = pd.DataFrame(rows).sort_values(["ks_alert", "psi"], ascending=[False, False]).reset_index(drop=True)
    return out

@dataclass(frozen=True)
class RunConfig:
    project_name: str = "dascient_deepgenai_metrics"
    run_name: str = "baseline"
    seed: int = 42
    output_dir: str = "./runs"
    jsonl_log_name: str = "metrics.jsonl"

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def stable_hash(self) -> str:
        blob = json.dumps(self.to_dict(), sort_keys=True).encode("utf-8")
        return hashlib.sha256(blob).hexdigest()[:16]

def jsonl_append(path: str, record: Dict[str, Any]) -> None:
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, sort_keys=True) + "\\n")
"""
from pathlib import Path

def write_text(path: str, text: str) -> None:
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)

core_path = str(Path(run_dir) / "metrics_core.py")
write_text(core_path, metrics_core_py.strip() + "\\n")
core_path



# Closing

This notebook gives you a working spine:

- deterministic runs  
- dataset hashes  
- drift detection  
- metric registry with versions and tests  
- robustness via controlled shift sets  
- JSONL logs for dashboards  
- model card and run report templates  
- an exportable module stub  

To extend this for deep and generative systems, swap the model adapter and add metrics in the registry.

Build instruments. Ship clarity.

DaScient
